In [1]:
"""
Soft-Vote Ensemble (SVM + XGBoost + LightGBM + LogReg) — SFS Loop  [v2]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Upgrades over v1
────────────────
  [1] Config dataclass        — all hyper-params & paths in one place
  [2] seed_everything()       — NumPy + Python reproducibility
  [3] Checkpoint / resume     — reads existing CSV rows, skips done steps
  [4] OOF-based Youden thr.   — uses CV probs, not in-sample → no leakage
  [5] Manual OOF CV loop      — replaces slow cross_val_predict(Voting…),
                                 same averaged probabilities, 4× more transparent
  [6] Early stopping XGB/LGBM — split-before-SMOTE val set (clean early stop)
  [7] Per-model AUC columns   — AUC_SVM, AUC_XGB, AUC_LGBM, AUC_LR
  [8] soft_vote() helper      — configurable per-model weights
  [9] Memory cleanup           — del models after each SFS step
 [10] tqdm progress bar        — live CV/Test AUC + threshold display
 [11] XGBoost deprecation fix  — use_label_encoder removed
 [12] os.makedirs safety       — auto-creates output dir
"""
from __future__ import annotations

import logging
import os
import random
import warnings
from dataclasses import dataclass

import lightgbm as lgb
import mygene
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score, roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from tqdm import tqdm
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)



In [2]:

# ══════════════════════════════════════════════════════════
# [1] CONFIG
# ══════════════════════════════════════════════════════════
@dataclass
class Config:
    seed: int = 42
    n_splits: int = 5
    sfs_start: int = 10        # inclusive; SFS begins at feature index 10
    n_features: int = 208      # total features in ranked pool
    smote_k: int = 3

    # ── SVM ───────────────────────────────────────────────
    svm_C: float = 1.0
    svm_gamma: str = "scale"

    # ── XGBoost ──────────────────────────────────────────
    xgb_n_est: int = 300
    xgb_depth: int = 4
    xgb_lr: float = 0.05
    xgb_early_stop: int = 20

    # ── LightGBM ─────────────────────────────────────────
    lgbm_n_est: int = 300
    lgbm_depth: int = 4
    lgbm_lr: float = 0.05
    lgbm_early_stop: int = 20

    # ── LogisticRegression ───────────────────────────────
    lr_C: float = 1.0
    lr_max_iter: int = 2000

    # ── [8] Soft-vote weights (must sum ≠ 0; auto-normalised) ──
    w_svm: float = 0.25
    w_xgb: float = 0.25
    w_lgbm: float = 0.25
    w_lr: float = 0.25

    # ── Paths ─────────────────────────────────────────────
    data_dir: str = (
        r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026"
        r"\Gene Expression Omnibus (GEO) Dataset GSE68086"
        r"\python_desque2_lung\Binary\data\ranked rf"
    )
    out_dir: str = (
        r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026"
        r"\Gene Expression Omnibus (GEO) Dataset GSE68086"
        r"\python_desque2_lung\Binary\data\sfs"
    )
    train_csv: str = "VotingClassifier_v2_train.csv"
    test_csv: str  = "VotingClassifier_v2_test.csv"


CFG = Config()



In [3]:

# ══════════════════════════════════════════════════════════
# [2] UTILITIES
# ══════════════════════════════════════════════════════════
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


def setup_logging() -> logging.Logger:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        datefmt="%H:%M:%S",
    )
    return logging.getLogger(__name__)


seed_everything(CFG.seed)
log = setup_logging()
os.makedirs(CFG.out_dir, exist_ok=True)  # [12]



In [4]:

# ══════════════════════════════════════════════════════════
# LOAD DATA + ENSEMBL → SYMBOL
# ══════════════════════════════════════════════════════════
X_train = pd.read_csv(os.path.join(CFG.data_dir, "X_train_rF_DEG_train_lung.csv"), low_memory=False)
X_test  = pd.read_csv(os.path.join(CFG.data_dir, "X_test_rF_DEG_train_lung.csv"),  low_memory=False)
y_train = pd.read_csv(os.path.join(CFG.data_dir, "y_train_rF_DEG_train.csv"), low_memory=False)
y_test  = pd.read_csv(os.path.join(CFG.data_dir, "y_test_rF_DEG_train.csv"),  low_memory=False)

ensembl_ids = [c for c in X_train.columns if c.startswith("ENSG")]
log.info(f"Querying MyGene.info for {len(ensembl_ids)} IDs…")
mg      = mygene.MyGeneInfo()
results = mg.querymany(ensembl_ids, scopes="ensembl.gene", fields="symbol", species="human")
id_map  = {r["query"]: r.get("symbol", r["query"]) for r in results}
X_train.rename(columns=id_map, inplace=True)
X_test.rename(columns=id_map, inplace=True)
X_train = X_train.iloc[:, :CFG.n_features]
X_test  = X_test.iloc[:, :CFG.n_features]
log.info("Gene symbol conversion complete.")

# ── Scale ─────────────────────────────────────────────────
scaler     = StandardScaler()
x_train_sc = scaler.fit_transform(X_train)
x_test_sc  = scaler.transform(X_test)

list_feature = list(X_train.columns)
ranks  = [[list_feature[j] for j in range(i + 1)] for i in range(len(list_feature))]
ratio  = float((y_train["cancer_type"] == 0).sum() / (y_train["cancer_type"] == 1).sum())
skf    = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed)
WEIGHTS = [CFG.w_svm, CFG.w_xgb, CFG.w_lgbm, CFG.w_lr]


# ══════════════════════════════════════════════════════════
# METRICS
# ══════════════════════════════════════════════════════════
def cfm(y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray) -> np.ndarray:
    auc = roc_auc_score(y_true, y_prob)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    acc  = accuracy_score(y_true, y_pred)
    rec  = tp / (tp + fn)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    spec = tn / (tn + fp)
    f1   = (2 * rec * prec) / (rec + prec) if (rec + prec) > 0 else 0.0
    # returns [acc, recall, precision, specificity, sensitivity, f1, auc]
    return np.array([acc, rec, prec, spec, rec, f1, auc])



00:04:02 [INFO] Querying MyGene.info for 211 IDs…
00:04:02 [INFO] querying 1-211 ...
00:04:04 [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
00:04:05 [INFO] Finished.
00:04:05 [WARNING] 5 input query terms found no hit:	['ENSG00000197459', 'ENSG00000267243', 'ENSG00000136653', 'ENSG00000163386', 'ENSG00000148362']
00:04:05 [INFO] Pass "returnall=True" to return complete lists of duplicate or missing query terms.
00:04:05 [INFO] Gene symbol conversion complete.


In [5]:

# ══════════════════════════════════════════════════════════
# [6] MODEL FACTORIES — split-before-SMOTE for clean early stopping
# ══════════════════════════════════════════════════════════
def _smote_split(X: np.ndarray, y: np.ndarray, val_frac: float = 0.15):
    """
    1. Split original data → train_s / val (no synthetic samples in val)
    2. Apply SMOTE only to train_s
    Returns: X_sm, y_sm, X_val, y_val
    """
    X_s, X_v, y_s, y_v = train_test_split(
        X, y, test_size=val_frac, stratify=y, random_state=CFG.seed
    )
    X_sm, y_sm = SMOTE(k_neighbors=CFG.smote_k, random_state=CFG.seed).fit_resample(X_s, y_s)
    return X_sm, y_sm, X_v, y_v


def fit_svm(X: np.ndarray, y: np.ndarray) -> SVC:
    # SVC has no early stopping → SMOTE full training data
    X_sm, y_sm = SMOTE(k_neighbors=CFG.smote_k, random_state=CFG.seed).fit_resample(X, y)
    m = SVC(
        kernel="rbf", C=CFG.svm_C, gamma=CFG.svm_gamma,
        class_weight="balanced", probability=True, random_state=CFG.seed,
    )
    m.fit(X_sm, y_sm)
    return m


def fit_xgb(X: np.ndarray, y: np.ndarray) -> XGBClassifier:  # [11] no use_label_encoder
    X_sm, y_sm, X_v, y_v = _smote_split(X, y)
    m = XGBClassifier(
        n_estimators=CFG.xgb_n_est, max_depth=CFG.xgb_depth, learning_rate=CFG.xgb_lr,
        scale_pos_weight=ratio, eval_metric="logloss",
        early_stopping_rounds=CFG.xgb_early_stop,
        random_state=CFG.seed, n_jobs=-1,
    )
    m.fit(X_sm, y_sm, eval_set=[(X_v, y_v)], verbose=False)
    return m


def fit_lgbm(X: np.ndarray, y: np.ndarray) -> LGBMClassifier:
    X_sm, y_sm, X_v, y_v = _smote_split(X, y)
    m = LGBMClassifier(
        n_estimators=CFG.lgbm_n_est, max_depth=CFG.lgbm_depth, learning_rate=CFG.lgbm_lr,
        is_unbalance=True, random_state=CFG.seed, n_jobs=-1, verbose=-1,
    )
    m.fit(
        X_sm, y_sm, eval_set=[(X_v, y_v)],
        callbacks=[
            lgb.early_stopping(CFG.lgbm_early_stop, verbose=False),
            lgb.log_evaluation(-1),
        ],
    )
    return m


def fit_lr(X: np.ndarray, y: np.ndarray) -> LogisticRegression:
    X_sm, y_sm = SMOTE(k_neighbors=CFG.smote_k, random_state=CFG.seed).fit_resample(X, y)
    m = LogisticRegression(
        solver="liblinear", penalty="l2", C=CFG.lr_C,
        class_weight="balanced", max_iter=CFG.lr_max_iter, random_state=CFG.seed,
    )
    m.fit(X_sm, y_sm)
    return m



In [6]:

# ══════════════════════════════════════════════════════════
# [8] SOFT VOTE HELPER
# ══════════════════════════════════════════════════════════
def soft_vote(probs: list[np.ndarray], weights: list[float]) -> np.ndarray:
    """Weighted average of probability arrays; weights are auto-normalised."""
    w = np.array(weights, dtype=float)
    w /= w.sum()
    return sum(p * wi for p, wi in zip(probs, w))


# ══════════════════════════════════════════════════════════
# [3] CHECKPOINT / RESUME
# ══════════════════════════════════════════════════════════
COLS = [
    "Gene", "ACC", "Recall", "Precision", "Sp", "Sn", "f1_Score", "AUC",
    "AUC_SVM", "AUC_XGB", "AUC_LGBM", "AUC_LR",
]
train_out = os.path.join(CFG.out_dir, CFG.train_csv)
test_out  = os.path.join(CFG.out_dir, CFG.test_csv)

result_train: list = []
result_test:  list = []
resume_from = CFG.sfs_start

if os.path.exists(train_out):
    _ck = pd.read_csv(train_out)
    if len(_ck) > 0:
        result_train = _ck.values.tolist()
        resume_from  = CFG.sfs_start + len(_ck)
        log.info(f"Resuming from feature index {resume_from} ({len(_ck)} steps already done).")

if os.path.exists(test_out):
    _ck2 = pd.read_csv(test_out)
    if len(_ck2) > 0:
        result_test = _ck2.values.tolist()



In [7]:

# ══════════════════════════════════════════════════════════
# [4][5][7][9][10] SFS LOOP
# ══════════════════════════════════════════════════════════
log.info("Starting Voting Classifier SFS loop…")
n_steps = len(ranks) - CFG.sfs_start

with tqdm(total=n_steps, desc="SFS", unit="step",
          initial=resume_from - CFG.sfs_start) as pbar:

    for i in range(CFG.sfs_start, len(ranks)):
        if i < resume_from:
            continue

        n_feat = i + 1
        X_tr   = x_train_sc[:, :n_feat]
        X_te   = x_test_sc[:, :n_feat]
        y_tr   = y_train["cancer_type"].values
        y_te   = y_test["cancer_type"].values

        # ── [5] Manual OOF CV — fit each model per fold ──────
        oof_svm  = np.zeros(len(y_tr))
        oof_xgb  = np.zeros(len(y_tr))
        oof_lgbm = np.zeros(len(y_tr))
        oof_lr   = np.zeros(len(y_tr))

        for tr_idx, val_idx in skf.split(X_tr, y_tr):
            Xf_tr, Xf_val = X_tr[tr_idx], X_tr[val_idx]
            yf_tr         = y_tr[tr_idx]

            svm_f  = fit_svm(Xf_tr, yf_tr)
            xgb_f  = fit_xgb(Xf_tr, yf_tr)
            lgbm_f = fit_lgbm(Xf_tr, yf_tr)
            lr_f   = fit_lr(Xf_tr, yf_tr)

            oof_svm[val_idx]  = svm_f.predict_proba(Xf_val)[:, 1]
            oof_xgb[val_idx]  = xgb_f.predict_proba(Xf_val)[:, 1]
            oof_lgbm[val_idx] = lgbm_f.predict_proba(Xf_val)[:, 1]
            oof_lr[val_idx]   = lr_f.predict_proba(Xf_val)[:, 1]
            del svm_f, xgb_f, lgbm_f, lr_f  # [9]

        # ── [8] Soft-vote OOF probabilities ──────────────────
        cv_probs = soft_vote([oof_svm, oof_xgb, oof_lgbm, oof_lr], WEIGHTS)

        # ── [4] OOF-based Youden threshold (no leakage) ──────
        fpr_cv, tpr_cv, thr_cv = roc_curve(y_tr, cv_probs)
        best_thr = float(thr_cv[np.argmax(tpr_cv - fpr_cv)])

        # ── CV metrics ───────────────────────────────────────
        y_cv_pred = (cv_probs >= 0.5).astype(int)
        cv_m      = cfm(y_tr, y_cv_pred, cv_probs)

        # ── Fit final models on full training data ────────────
        svm_final  = fit_svm(X_tr, y_tr)
        xgb_final  = fit_xgb(X_tr, y_tr)
        lgbm_final = fit_lgbm(X_tr, y_tr)
        lr_final   = fit_lr(X_tr, y_tr)

        # ── Test prediction with locked Youden threshold ──────
        p_svm_te  = svm_final.predict_proba(X_te)[:, 1]
        p_xgb_te  = xgb_final.predict_proba(X_te)[:, 1]
        p_lgbm_te = lgbm_final.predict_proba(X_te)[:, 1]
        p_lr_te   = lr_final.predict_proba(X_te)[:, 1]
        p_vote_te = soft_vote([p_svm_te, p_xgb_te, p_lgbm_te, p_lr_te], WEIGHTS)

        y_pred_te = (p_vote_te >= best_thr).astype(int)
        te_m      = cfm(y_te, y_pred_te, p_vote_te)

        del svm_final, xgb_final, lgbm_final, lr_final  # [9]

        # ── [7] Save with per-model AUC columns ──────────────
        result_train.append([
            ranks[i],
            *[f"{v * 100:.2f}%" for v in cv_m],
            f"{roc_auc_score(y_tr, oof_svm)  * 100:.2f}%",
            f"{roc_auc_score(y_tr, oof_xgb)  * 100:.2f}%",
            f"{roc_auc_score(y_tr, oof_lgbm) * 100:.2f}%",
            f"{roc_auc_score(y_tr, oof_lr)   * 100:.2f}%",
        ])
        pd.DataFrame(result_train, columns=COLS).to_csv(train_out, index=False)

        result_test.append([
            ranks[i],
            *[f"{v * 100:.2f}%" for v in te_m],
            f"{roc_auc_score(y_te, p_svm_te)  * 100:.2f}%",
            f"{roc_auc_score(y_te, p_xgb_te)  * 100:.2f}%",
            f"{roc_auc_score(y_te, p_lgbm_te) * 100:.2f}%",
            f"{roc_auc_score(y_te, p_lr_te)   * 100:.2f}%",
        ])
        pd.DataFrame(result_test, columns=COLS).to_csv(test_out, index=False)

        # ── [10] tqdm postfix ─────────────────────────────────
        pbar.set_postfix(
            n=n_feat,
            CV_AUC=f"{cv_m[6] * 100:.1f}%",
            Te_AUC=f"{te_m[6] * 100:.1f}%",
            thr=f"{best_thr:.3f}",
        )
        pbar.update(1)

log.info("Done! Results saved.")
df_tr = pd.read_csv(train_out)
df_te = pd.read_csv(test_out)
print("\nFinal 5 training rows:\n", df_tr.tail(5).to_string(index=False))
print("\nFinal 5 test rows:\n",     df_te.tail(5).to_string(index=False))

00:04:05 [INFO] Starting Voting Classifier SFS loop…
SFS: 100%|██████████| 198/198 [1:05:33<00:00, 19.87s/step, CV_AUC=96.8%, Te_AUC=95.6%, n=208, thr=0.442]
01:09:39 [INFO] Done! Results saved.



Final 5 training rows:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                